# Pareto distribution and power laws
The Pareto PDF and CDF, the 80-20 rule, and two checks for Pareto data: a log-log plot and a Q-Q plot.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# fixed seed: the same random data on every run
rng = np.random.default_rng(42)

## 1. A power law
y = k x^a with k = 1 and a = -2: doubling x divides y by 4.

In [ ]:
k, a = 1, -2
for x in [1, 2, 4, 10]:
    print(x, k * x ** a)

## 2. The Pareto PDF and CDF
In scipy, `b` is alpha and `scale` is x_m.

In [ ]:
p = stats.pareto(b=3, scale=1)
print("f(1) =", p.pdf(1), " f(2) =", p.pdf(2))
print("P(X > 2) =", p.sf(2), " F(2) =", p.cdf(2))
x = np.linspace(0.5, 5, 500)
fig = make_subplots(rows=1, cols=2, subplot_titles=["PDF", "CDF"])
for alpha in [1, 2, 3]:
    d = stats.pareto(alpha)
    fig.add_scatter(x=x, y=d.pdf(x), name=f"alpha = {alpha}", row=1, col=1)
    fig.add_scatter(x=x, y=d.cdf(x), name=f"alpha = {alpha}", showlegend=False, row=1, col=2)
fig.update_layout(template="simple_white")
fig.show()

## 3. The 80-20 rule
The richest fraction p holds p^(1 - 1/alpha) of the total.

In [ ]:
alpha_8020 = np.log(5) / np.log(4)
print("alpha for 80-20:", round(alpha_8020, 3))
for alpha in [alpha_8020, 2, 3]:
    print(f"alpha {alpha:.2f}: top 20% hold {0.2 ** (1 - 1 / alpha):.0%}")

In [ ]:
# check by simulation: 1,000,000 wealth values with alpha = 1.16
w = np.sort(stats.pareto.rvs(alpha_8020, size=1_000_000, random_state=rng))
top20 = w[int(0.8 * len(w)):].sum() / w.sum()
# heavy tail: the simulated share wobbles around 80% from run to run
print("simulated share of the top 20%:", round(top20, 3))

## 4. Is it Pareto? Log-log plot
The PDF formula on log-log axes is a straight line with slope -(alpha + 1). For data, plot the log of the share of values at or above each x.

In [ ]:
# a fresh generator with seed 42, as in the Note figure
rng2 = np.random.default_rng(42)
par = stats.pareto.rvs(3, size=1000, random_state=rng2)
logn = rng2.lognormal(0.3, 0.5, 1000)
fig = go.Figure()
for data, name in [(par, "Pareto data"), (logn, "log-normal data")]:
    s = np.sort(data)
    share_above = 1 - np.arange(len(s)) / len(s)
    fig.add_scatter(x=np.log(s), y=np.log(share_above), mode="markers", marker_size=3, name=name)
fig.update_layout(template="simple_white", xaxis_title="ln x", yaxis_title="ln(share of values >= x)")
fig.show()
# slope of the straight part for the Pareto data: about -alpha
s = np.sort(par)
share_above = 1 - np.arange(len(s)) / len(s)
print("fitted slope:", round(np.polyfit(np.log(s[:950]), np.log(share_above[:950]), 1)[0], 2))

## 5. Is it Pareto? Q-Q plot against a fitted Pareto

In [ ]:
b, loc, scale = stats.pareto.fit(par, floc=0)
print("fitted alpha:", round(b, 2), " x_m:", round(scale, 3))
(osm, osr), (slope, icpt, r) = stats.probplot(par, dist=stats.pareto, sparams=(b, loc, scale))
fig = go.Figure()
fig.add_scatter(x=osm, y=osr, mode="markers", name="data")
fig.add_scatter(x=osm, y=slope * osm + icpt, mode="lines", name="line")
fig.update_layout(template="simple_white", xaxis_title="Pareto quantiles", yaxis_title="data quantiles")
fig.show()

## 6. Why the last Q-Q points drift
Draw 500 fresh samples of 1,000 Pareto values and measure how much each sorted position varies across samples.

In [ ]:
r = np.random.default_rng(0)
S = np.sort(stats.pareto.rvs(3, size=(500, 1000), random_state=r), axis=1)
for k in [500, 900, 990, 999, 1000]:
    q = S[:, k - 1]
    print(f"value #{k}: mean {q.mean():.2f}, spread (std/mean) {q.std() / q.mean():.0%}")
